In [2]:
import os
import re
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

print("train:", train.shape)
print("train_series:", train_series.shape)
print("test:", test.shape)
print("test_series:", test_series.shape)

train: (4407, 14)
train_series: (24371, 5)
test: (3, 1)
test_series: (15, 5)


In [3]:
gold_mask = train[TARGETS].notna().all(axis=1)

gold = train[gold_mask].copy()
unlabeled = train[~gold_mask].copy()

print("Gold labelled studies:", len(gold))
print("Unlabelled studies:", len(unlabeled))

print("\nGold label prevalence:")
display(gold[TARGETS].mean().sort_values())

Gold labelled studies: 58
Unlabelled studies: 4349

Gold label prevalence:


MCL                 0.155172
Lateral OA          0.189655
Baker's             0.206897
Medial OA           0.258621
Fracture            0.310345
Contusion           0.327586
PF OA               0.362069
Lateral Meniscus    0.396552
ACL                 0.413793
Medial Meniscus     0.448276
Synovitis           0.465517
Effusion            0.603448
dtype: float64

In [4]:
def normalize_report(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # normalize common punctuation
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


train["report_clean"] = train["Report"].apply(normalize_report)

In [5]:
POSITIVE_PATTERNS = {

    "ACL": [
        r"acl.{0,80}(tear|ruptur|injur|absent|discontinu)",
        r"(tear|ruptur|injur).{0,80}acl",
        r"anterior cruciate.{0,80}(tear|ruptur|injur)",
    ],

    "MCL": [
        r"mcl.{0,80}(tear|ruptur|injur|sprain)",
        r"(tear|ruptur|injur|sprain).{0,80}mcl",
        r"medial collateral.{0,80}(tear|ruptur|injur|sprain)",
    ],

    "Medial Meniscus": [
        r"medial menisc.{0,80}(tear|tear|ruptur|injur)",
        r"(tear|ruptur|injur).{0,80}medial menisc",
    ],

    "Lateral Meniscus": [
        r"lateral menisc.{0,80}(tear|ruptur|injur)",
        r"(tear|ruptur|injur).{0,80}lateral menisc",
    ],

    "Medial OA": [
        r"medial.{0,100}(osteoarthritis|arthrosis|degenerative)",
        r"medial compartment.{0,100}(cartilage loss|cartilage thinning|joint space)",
        r"medial tibiofemoral.{0,100}(degenerative|arthr)",
    ],

    "Lateral OA": [
        r"lateral.{0,100}(osteoarthritis|arthrosis|degenerative)",
        r"lateral compartment.{0,100}(cartilage loss|cartilage thinning|joint space)",
        r"lateral tibiofemoral.{0,100}(degenerative|arthr)",
    ],

    "PF OA": [
        r"patellofemoral.{0,100}(osteoarthritis|arthrosis|degenerative)",
        r"patellofemoral.{0,100}(cartilage loss|cartilage thinning)",
        r"patellar.{0,80}(cartilage loss|cartilage thinning)",
    ],

    "Effusion": [
        r"(small|moderate|large|mild|moderate|marked).{0,30}joint effusion",
        r"joint effusion",
        r"knee effusion",
        r"effusion",
    ],

    "Synovitis": [
        r"synovitis",
        r"synovial.{0,50}(thickening|hypertrophy|inflammation)",
    ],

    "Baker's": [
        r"baker.?s cyst",
        r"popliteal cyst",
    ],

    "Contusion": [
        r"bone contusion",
        r"bone bruise",
        r"osseous contusion",
        r"bone marrow edema",
        r"marrow edema",
    ],

    "Fracture": [
        r"fracture",
        r"fractured",
        r"fracture line",
    ],
}

NEGATION_PATTERNS = [
    r"\bno\b",
    r"\bnot\b",
    r"\bwithout\b",
    r"\bnegative for\b",
    r"\bno evidence of\b",
    r"\babsence of\b",
    r"\bfree of\b",
    r"\bintact\b",
    r"\bpreserved\b",
    r"\bunremarkable\b",
    r"\bnormal\b",
]

In [6]:
def negated(text, start, end):

    window_before = text[max(0, start - 120):start]
    window_after = text[end:min(len(text), end + 80)]

    for p in NEGATION_PATTERNS:
        if re.search(p, window_before):
            return True

    for p in [
        r"\bintact\b",
        r"\bnormal\b",
        r"\bunremarkable\b",
        r"\bpreserved\b",
        r"\bno tear\b",
        r"\bno injury\b"
    ]:
        if re.search(p, window_after):
            return True

    return False

In [7]:
def extract_report_labels(report):

    report = normalize_report(report)

    output = {}

    for target, patterns in POSITIVE_PATTERNS.items():

        positive = False
        negative = False

        for pattern in patterns:

            matches = list(
                re.finditer(
                    pattern,
                    report,
                    flags=re.IGNORECASE
                )
            )

            for m in matches:

                if negated(
                    report,
                    m.start(),
                    m.end()
                ):
                    negative = True
                else:
                    positive = True

        if positive:
            output[target] = 1.0

        elif negative:
            output[target] = 0.0

        else:
            output[target] = np.nan

    return output


In [8]:
parsed = train["Report"].apply(
    extract_report_labels
)

parsed = pd.DataFrame(
    parsed.tolist(),
    index=train.index
)

display(parsed.notna().sum())
display(parsed.mean())

ACL                  701
MCL                  583
Medial Meniscus      857
Lateral Meniscus     886
Medial OA            260
Lateral OA           184
PF OA                226
Effusion            1572
Synovitis            304
Baker's              844
Contusion            877
Fracture             643
dtype: int64

ACL                 0.423680
MCL                 0.310463
Medial Meniscus     0.522754
Lateral Meniscus    0.351016
Medial OA           0.638462
Lateral OA          0.592391
PF OA               0.495575
Effusion            0.370865
Synovitis           0.868421
Baker's             0.242891
Contusion           0.452680
Fracture            0.167963
dtype: float64

In [9]:
from sklearn.metrics import roc_auc_score

print("REPORT LABEL PERFORMANCE")
print("=" * 70)

for target in TARGETS:

    mask = (
        train.loc[gold.index, target].notna()
        &
        parsed.loc[gold.index, target].notna()
    )

    if mask.sum() < 5:
        continue

    y_true = train.loc[
        gold.index[mask],
        target
    ].astype(int)

    y_pred = parsed.loc[
        gold.index[mask],
        target
    ]

    auc = roc_auc_score(
        y_true,
        y_pred
    )

    print(
        f"{target:20s} "
        f"N={mask.sum():2d} "
        f"AUC={auc:.3f}"
    )

REPORT LABEL PERFORMANCE
ACL                  N=18 AUC=0.556
MCL                  N=10 AUC=0.583
Medial Meniscus      N=16 AUC=0.764
Lateral Meniscus     N=24 AUC=0.762
PF OA                N= 7 AUC=0.542
Effusion             N=28 AUC=0.685
Synovitis            N= 9 AUC=0.417
Baker's              N= 9 AUC=1.000
Contusion            N=19 AUC=0.461
Fracture             N=14 AUC=0.929


In [10]:
series = train_series.copy()

series["plane"] = (
    series["Anatomical_Plane"]
    .astype(str)
    .str.lower()
)

series["fluid"] = (
    series["Fluid_Sensitive"]
    .astype(str)
    .str.lower()
)

print(series["plane"].value_counts())
print(series["fluid"].value_counts())

display(series.head(20))

plane
sagittal    9864
coronal     8609
axial       5898
Name: count, dtype: int64
fluid
1    14010
0    10361
Name: count, dtype: int64


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane,plane,fluid
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal,sagittal,1
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial,axial,1
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,0,0,Coronal,coronal,0
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal,coronal,1
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,0,0,Sagittal,sagittal,0
5,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.27845969797953115400...,0,0,Sagittal,sagittal,0
6,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.72137122913943091996...,1,1,Sagittal,sagittal,1
7,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.83327574802189551711...,1,1,Axial,axial,1
8,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.87374594163896635184...,0,0,Coronal,coronal,0
9,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.99023014295026495734...,1,1,Coronal,coronal,1


In [11]:
def choose_series(group):

    selected = []

    # Prefer fluid-sensitive series
    fluid = group[
        group["Fluid_Sensitive"].astype(str).str.lower()
        .isin(["true", "1", "yes"])
    ]

    if len(fluid) == 0:
        fluid = group

    for plane in ["sagittal", "coronal", "axial"]:

        candidates = fluid[
            fluid["Anatomical_Plane"]
            .astype(str)
            .str.lower()
            .str.contains(plane, na=False)
        ]

        if len(candidates):
            selected.append(
                candidates.iloc[0]
            )

    # fallback if plane names don't match
    if len(selected) == 0:
        selected.append(fluid.iloc[0])

    return pd.DataFrame(selected)

selected_series = (
    series
    .groupby("StudyInstanceUID", group_keys=False)
    .apply(choose_series)
    .reset_index(drop=True)
)

print(selected_series.shape)
display(selected_series.head())

(12805, 7)


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane,plane,fluid
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal,sagittal,1
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal,coronal,1
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial,axial,1
3,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.72137122913943091996...,1,1,Sagittal,sagittal,1
4,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.99023014295026495734...,1,1,Coronal,coronal,1


In [12]:
LABEL_CONFIDENCE = {
    "ACL": 1.0,
    "MCL": 1.0,
    "Medial Meniscus": 1.0,
    "Lateral Meniscus": 1.0,

    "Medial OA": 0.7,
    "Lateral OA": 0.7,
    "PF OA": 0.7,

    "Effusion": 0.8,
    "Synovitis": 0.6,

    "Baker's": 1.0,
    "Contusion": 0.7,
    "Fracture": 1.0
}

In [13]:
labels = pd.DataFrame(
    np.nan,
    index=train.index,
    columns=TARGETS
)

weights = pd.DataFrame(
    0.0,
    index=train.index,
    columns=TARGETS
)

for target in TARGETS:

    # Gold labels always override weak labels
    gold_available = train[target].notna()

    labels.loc[gold_available, target] = (
        train.loc[gold_available, target]
    )

    weights.loc[gold_available, target] = 1.0

    # Weak labels for remaining studies
    weak_available = (
        ~gold_available
        & parsed[target].notna()
    )

    labels.loc[weak_available, target] = (
        parsed.loc[weak_available, target]
    )

    weights.loc[weak_available, target] = (
        LABEL_CONFIDENCE[target]
    )

print("Usable labels per target:")
display(labels.notna().sum())

Usable labels per target:


ACL                  741
MCL                  631
Medial Meniscus      899
Lateral Meniscus     920
Medial OA            314
Lateral OA           239
PF OA                277
Effusion            1602
Synovitis            353
Baker's              893
Contusion            916
Fracture             687
dtype: int64

In [14]:
import pydicom
import numpy as np
from pathlib import Path
from functools import lru_cache

def get_dicom_files(series_path):
    return sorted(
        Path(series_path).glob("*.dcm"),
        key=lambda p: p.name
    )


def load_series(series_path, max_slices=16):
    """
    Load a DICOM series and uniformly sample up to max_slices.
    Returns:
        volume: [N, H, W], float32
    """

    files = get_dicom_files(series_path)

    if len(files) == 0:
        return None

    slices = []

    for f in files:
        try:
            ds = pydicom.dcmread(f)

            img = ds.pixel_array.astype(np.float32)

            # Apply rescale if available
            slope = float(getattr(ds, "RescaleSlope", 1.0))
            intercept = float(getattr(ds, "RescaleIntercept", 0.0))

            img = img * slope + intercept

            slices.append((ds, img))

        except Exception as e:
            print(f"Failed: {f}")
            print(e)

    if len(slices) == 0:
        return None

    # Try to order anatomically
    try:
        slices.sort(
            key=lambda x: float(
                x[0].ImagePositionPatient[2]
            )
        )
    except Exception:
        pass

    images = [x[1] for x in slices]

    # Uniform sampling
    if len(images) > max_slices:
        indices = np.linspace(
            0,
            len(images) - 1,
            max_slices
        ).astype(int)

        images = [images[i] for i in indices]

    volume = np.stack(images).astype(np.float32)

    return volume

In [15]:
def normalize_mri(img):

    img = img.astype(np.float32)

    # Remove extreme background
    nonzero = img[img != 0]

    if len(nonzero) == 0:
        return np.zeros_like(img)

    lo = np.percentile(nonzero, 1)
    hi = np.percentile(nonzero, 99)

    img = np.clip(img, lo, hi)

    img = (img - lo) / (hi - lo + 1e-6)

    return img

import cv2

IMG_SIZE = 224

def resize_volume(volume):

    output = []

    for img in volume:

        img = normalize_mri(img)

        img = cv2.resize(
            img,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_AREA
        )

        output.append(img)

    return np.stack(output).astype(np.float32)

In [16]:
def select_series_for_study(
    study_id,
    series_df,
    max_per_plane=1
):

    df = series_df[
        series_df["StudyInstanceUID"] == study_id
    ].copy()

    selected = []

    for plane in ["Sagittal", "Coronal", "Axial"]:

        candidates = df[
            df["Anatomical_Plane"]
            .astype(str)
            .str.lower()
            == plane.lower()
        ]

        if len(candidates) == 0:
            continue

        # Prefer fluid-sensitive
        fluid = candidates[
            candidates["Fluid_Sensitive"] == 1
        ]

        if len(fluid) > 0:
            candidates = fluid

        selected.append(
            candidates.iloc[0]
        )

    return selected

In [17]:
import numpy as np
from pathlib import Path

PLANE_ORDER = ["Sagittal", "Coronal", "Axial"]
SLICES_PER_SERIES = 5
IMG_SIZE = 224


def load_study(study_id, series_df, base_dir):

    df = series_df[
        series_df["StudyInstanceUID"] == study_id
    ].copy()

    output = []

    for plane in PLANE_ORDER:

        candidates = df[
            df["Anatomical_Plane"]
            .astype(str)
            .str.lower()
            == plane.lower()
        ]

        # No series for this plane
        if len(candidates) == 0:
            output.append(
                np.zeros(
                    (SLICES_PER_SERIES, IMG_SIZE, IMG_SIZE),
                    dtype=np.float32
                )
            )
            continue

        # Prefer fluid-sensitive
        fluid = candidates[
            candidates["Fluid_Sensitive"] == 1
        ]

        if len(fluid) > 0:
            candidates = fluid

        row = candidates.iloc[0]

        series_id = row["SeriesInstanceUID"]

        series_path = (
            base_dir
            / "train_series"
            / study_id
            / series_id
        )

        volume = load_series(
            series_path,
            max_slices=SLICES_PER_SERIES
        )

        if volume is None:
            output.append(
                np.zeros(
                    (SLICES_PER_SERIES, IMG_SIZE, IMG_SIZE),
                    dtype=np.float32
                )
            )
            continue

        volume = resize_volume(volume)

        # Pad if fewer than 5 slices
        if len(volume) < SLICES_PER_SERIES:

            pad = np.zeros(
                (
                    SLICES_PER_SERIES - len(volume),
                    IMG_SIZE,
                    IMG_SIZE
                ),
                dtype=np.float32
            )

            volume = np.concatenate(
                [volume, pad],
                axis=0
            )

        # Safety
        volume = volume[:SLICES_PER_SERIES]

        output.append(volume)

    # Always:
    # [3 planes, 5 slices, 224, 224]
    return np.stack(output).astype(np.float32)

In [18]:
study_id = train.iloc[0]["StudyInstanceUID"]

x = load_study(
    study_id,
    train_series,
    DATA_DIR
)

print("Shape:", x.shape)
print("Min:", x.min())
print("Max:", x.max())

for i, plane in enumerate(PLANE_ORDER):
    print(
        plane,
        "non-zero:",
        np.count_nonzero(x[i])
    )

Shape: (3, 5, 224, 224)
Min: 0.0
Max: 1.0
Sagittal non-zero: 226610
Coronal non-zero: 198525
Axial non-zero: 197181


In [19]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# --------------------------------------------------
# 1. ALWAYS create a fresh pretrained ResNet18
# --------------------------------------------------

weights = models.ResNet18_Weights.DEFAULT

backbone = models.resnet18(
    weights=weights
)

# Save the ORIGINAL 3-channel convolution
old_conv = backbone.conv1

print("Original conv:", old_conv.weight.shape)

# --------------------------------------------------
# 2. Create 5-channel convolution
# --------------------------------------------------

new_conv = nn.Conv2d(
    in_channels=5,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

# --------------------------------------------------
# 3. Initialize from pretrained RGB weights
# --------------------------------------------------

with torch.no_grad():

    # First 3 channels = pretrained RGB weights
    new_conv.weight[:, :3, :, :] = old_conv.weight

    # Last 2 channels = mean of pretrained RGB weights
    mean_weight = old_conv.weight.mean(
        dim=1,
        keepdim=True
    )

    new_conv.weight[:, 3:, :, :] = mean_weight.repeat(
        1, 2, 1, 1
    )

# --------------------------------------------------
# 4. Replace first convolution
# --------------------------------------------------

backbone.conv1 = new_conv

print("New conv:", backbone.conv1.weight.shape)

# --------------------------------------------------
# 5. Remove original classifier
# --------------------------------------------------

feature_dim = backbone.fc.in_features

backbone.fc = nn.Identity()

print("Feature dimension:", feature_dim)

backbone = backbone.to(device)

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s] 


Original conv: torch.Size([64, 3, 7, 7])
New conv: torch.Size([64, 5, 7, 7])
Feature dimension: 512


In [20]:
class KneeModel(nn.Module):

    def __init__(self, backbone, feature_dim, n_targets=12):

        super().__init__()

        self.backbone = backbone

        self.classifier = nn.Sequential(

            nn.Linear(
                feature_dim * 3,
                512
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                512,
                n_targets
            )
        )

    def forward(self, x):

        # x:
        # [B, 3, 5, H, W]

        features = []

        for plane in range(3):

            plane_x = x[:, plane]

            f = self.backbone(
                plane_x
            )

            features.append(f)

        features = torch.cat(
            features,
            dim=1
        )

        return self.classifier(features)

In [21]:
model = KneeModel(
    backbone=backbone,
    feature_dim=feature_dim,
    n_targets=len(TARGETS)
)

model = model.to(device)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
)

Parameters: 11.98M


In [22]:
print(backbone.conv1.weight.shape)
print(feature_dim)
print(sum(p.numel() for p in model.parameters()))

torch.Size([64, 5, 7, 7])
512
11975884


In [23]:
from tqdm.auto import tqdm
import time
import shutil

CACHE_DIR = Path("/kaggle/working/mri_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Don't accidentally reuse old partial cache
for f in CACHE_DIR.glob("*.npy"):
    f.unlink()

benchmark_ids = train["StudyInstanceUID"].iloc[:20].tolist()

start = time.time()

successful = 0

for study_id in tqdm(benchmark_ids):

    try:

        volume = load_study(
            study_id,
            train_series,
            DATA_DIR
        )

        np.save(
            CACHE_DIR / f"{study_id}.npy",
            volume.astype(np.float16)
        )

        successful += 1

    except Exception as e:

        print(
            f"\nERROR for {study_id}:"
        )
        print(e)

elapsed = time.time() - start

per_study = elapsed / len(benchmark_ids)

print()
print("=" * 60)
print(f"Successful: {successful}/{len(benchmark_ids)}")
print(f"20-study time: {elapsed:.2f} sec")
print(f"Time/study: {per_study:.2f} sec")
print(
    f"Estimated 4407-study time: "
    f"{per_study * len(train) / 60:.1f} min"
)
print("=" * 60)

  0%|          | 0/20 [00:00<?, ?it/s]


Successful: 20/20
20-study time: 30.20 sec
Time/study: 1.51 sec
Estimated 4407-study time: 110.9 min


In [24]:
from tqdm.auto import tqdm
import time
import json

CACHE_DIR = Path("/kaggle/working/mri_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Make sure our cache contains only valid .npy files
existing = {
    p.stem for p in CACHE_DIR.glob("*.npy")
}

print(f"Already cached: {len(existing)}")

remaining = [
    sid for sid in train["StudyInstanceUID"]
    if sid not in existing
]

print(f"Remaining: {len(remaining)}")

start = time.time()

failed = []

for study_id in tqdm(
    remaining,
    desc="Preprocessing MRI"
):

    try:

        volume = load_study(
            study_id,
            train_series,
            DATA_DIR
        )

        np.save(
            CACHE_DIR / f"{study_id}.npy",
            volume.astype(np.float16)
        )

    except Exception as e:

        failed.append({
            "StudyInstanceUID": study_id,
            "error": str(e)
        })

elapsed = time.time() - start

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)

print(f"Cached: {len(list(CACHE_DIR.glob('*.npy')))}")
print(f"Failed: {len(failed)}")
print(f"Time: {elapsed / 60:.2f} minutes")

if failed:
    print("\nFirst failures:")
    for x in failed[:10]:
        print(x)

# Save failures
with open(
    "/kaggle/working/preprocessing_failures.json",
    "w"
) as f:
    json.dump(failed, f, indent=2)

Already cached: 20
Remaining: 4387


Preprocessing MRI:   0%|          | 0/4387 [00:00<?, ?it/s]


PREPROCESSING COMPLETE
Cached: 4407
Failed: 0
Time: 107.56 minutes


In [28]:
print("Cache:", len(list(CACHE_DIR.glob("*.npy"))))

missing = [
    sid for sid in train["StudyInstanceUID"]
    if not (CACHE_DIR / f"{sid}.npy").exists()
]

print("Missing:", len(missing))

if len(missing) > 0:
    print("First missing IDs:", missing[:10])

# Inspect one cached study
sample_cache = next(CACHE_DIR.glob("*.npy"))
arr = np.load(sample_cache)

print("Sample:", sample_cache.name)
print("Shape:", arr.shape)
print("Dtype:", arr.dtype)
print("Min:", arr.min(), "Max:", arr.max())

Cache: 4407
Missing: 0
Sample: 1.2.826.0.1.3680043.8.498.48033467661219494489168041469309669510.npy
Shape: (3, 5, 224, 224)
Dtype: float16
Min: 0.0 Max: 1.0


In [31]:
# Recreate weak-label confidence weights with a non-conflicting name

LABEL_CONFIDENCE = {
    "ACL": 1.0,
    "MCL": 1.0,
    "Medial Meniscus": 1.0,
    "Lateral Meniscus": 1.0,
    "Medial OA": 0.7,
    "Lateral OA": 0.7,
    "PF OA": 0.7,
    "Effusion": 0.8,
    "Synovitis": 0.6,
    "Baker's": 1.0,
    "Contusion": 0.7,
    "Fracture": 1.0
}

label_weights = pd.DataFrame(
    0.0,
    index=train.index,
    columns=TARGETS
)

# Gold labels get full confidence
for target in TARGETS:
    gold_available = train[target].notna()
    label_weights.loc[gold_available, target] = 1.0

    # Parsed report labels get target-specific confidence
    weak_available = (
        ~gold_available &
        parsed[target].notna()
    )

    label_weights.loc[weak_available, target] = LABEL_CONFIDENCE[target]

print("Label weights rebuilt.")
display(label_weights.head())

Label weights rebuilt.


,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.8,0.0,0.0,0.7,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [34]:
training_table = train[["StudyInstanceUID"]].copy()

for target in TARGETS:
    training_table[f"label_{target}"] = labels[target]
    training_table[f"weight_{target}"] = label_weights[target]

print("Training table:", training_table.shape)

display(training_table.head())

Training table: (4407, 25)


,StudyInstanceUID,label_ACL,weight_ACL,label_MCL,weight_MCL,label_Medial Meniscus,weight_Medial Meniscus,label_Lateral Meniscus,weight_Lateral Meniscus,label_Medial OA,...,label_Effusion,weight_Effusion,label_Synovitis,weight_Synovitis,label_Baker's,weight_Baker's,label_Contusion,weight_Contusion,label_Fracture,weight_Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,NaN,0.0,NaN,0.0,NaN,0.0,0.0,1.0,NaN,...,0.0,0.8,NaN,0.0,NaN,0.0,0.0,0.7,0.0,1.0
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,0.0,1.0


In [35]:
from sklearn.model_selection import train_test_split

gold_indices = gold.index.to_numpy()

gold_train_idx, gold_val_idx = train_test_split(
    gold_indices,
    test_size=0.20,
    random_state=SEED
)

val_indices = set(gold_val_idx)

train_indices = [
    idx for idx in train.index
    if idx not in val_indices
]

train_df = training_table.loc[train_indices].copy()
val_df = training_table.loc[gold_val_idx].copy()

print("Training studies:", len(train_df))
print("Validation studies:", len(val_df))

print("\nValidation label coverage:")
print(
    val_df[
        [f"label_{t}" for t in TARGETS]
    ].notna().sum()
)

Training studies: 4395
Validation studies: 12

Validation label coverage:
label_ACL                 12
label_MCL                 12
label_Medial Meniscus     12
label_Lateral Meniscus    12
label_Medial OA           12
label_Lateral OA          12
label_PF OA               12
label_Effusion            12
label_Synovitis           12
label_Baker's             12
label_Contusion           12
label_Fracture            12
dtype: int64


In [36]:
from torch.utils.data import Dataset, DataLoader

class KneeDataset(Dataset):
    def __init__(self, dataframe, cache_dir, targets, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.cache_dir = Path(cache_dir)
        self.targets = targets
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row["StudyInstanceUID"]

        # Cached MRI: [3 planes, 5 slices, 224, 224]
        x = np.load(
            self.cache_dir / f"{study_id}.npy"
        ).astype(np.float32)

        # Convert to tensor
        x = torch.from_numpy(x)

        # Simple horizontal flip augmentation
        if self.augment and random.random() < 0.5:
            x = torch.flip(x, dims=[-1])

        # Labels
        y = np.array(
            [row[f"label_{target}"] for target in self.targets],
            dtype=np.float32
        )

        # Confidence weights
        w = np.array(
            [row[f"weight_{target}"] for target in self.targets],
            dtype=np.float32
        )

        # Which labels actually exist?
        mask = ~np.isnan(y)

        # BCE cannot handle NaN
        y = np.nan_to_num(y, nan=0.0)

        return (
            x,
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(w, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool)
        )

In [37]:
BATCH_SIZE = 4

train_dataset = KneeDataset(
    train_df,
    CACHE_DIR,
    TARGETS,
    augment=True
)

val_dataset = KneeDataset(
    val_df,
    CACHE_DIR,
    TARGETS,
    augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train studies:", len(train_dataset))
print("Train batches:", len(train_loader))
print("Val studies:", len(val_dataset))
print("Val batches:", len(val_loader))

Train studies: 4395
Train batches: 1099
Val studies: 12
Val batches: 3


In [38]:
x, y, w, mask = next(iter(train_loader))

print("X:", x.shape, x.dtype)
print("Y:", y.shape, y.dtype)
print("W:", w.shape, w.dtype)
print("Mask:", mask.shape, mask.dtype)

print("\nX range:", x.min().item(), "to", x.max().item())
print("Valid labels in batch:", mask.sum().item())
print("Positive labels in batch:", y[mask].sum().item())

X: torch.Size([4, 3, 5, 224, 224]) torch.float32
Y: torch.Size([4, 12]) torch.float32
W: torch.Size([4, 12]) torch.float32
Mask: torch.Size([4, 12]) torch.bool

X range: 0.0 to 1.0
Valid labels in batch: 0
Positive labels in batch: 0.0


In [39]:
model.eval()

x_gpu = x.to(device, non_blocking=True)

with torch.no_grad():
    logits = model(x_gpu)

print("Logits shape:", logits.shape)
print("Logit min:", logits.min().item())
print("Logit max:", logits.max().item())

probs = torch.sigmoid(logits)

print("\nProbability shape:", probs.shape)
print("Probability min:", probs.min().item())
print("Probability max:", probs.max().item())

Logits shape: torch.Size([4, 12])
Logit min: -0.5369579792022705
Logit max: 0.6675837635993958

Probability shape: torch.Size([4, 12])
Probability min: 0.368895560503006
Probability max: 0.660961925983429


In [41]:
import torch.nn.functional as F

def weighted_masked_bce(logits, targets, weights, mask):
    """
    BCE loss that:
    - ignores missing labels
    - gives gold labels full weight
    - gives weak report labels reduced weight
    """
    loss = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none"
    )

    effective_weight = weights * mask.float()

    loss = loss * effective_weight

    denominator = effective_weight.sum().clamp(min=1.0)

    return loss.sum() / denominator

In [42]:
y_gpu = y.to(device, non_blocking=True)
w_gpu = w.to(device, non_blocking=True)
mask_gpu = mask.to(device, non_blocking=True)

loss = weighted_masked_bce(
    logits,
    y_gpu,
    w_gpu,
    mask_gpu
)

print("Sanity-check loss:", loss.item())
print("Loss is finite:", torch.isfinite(loss).item())

Sanity-check loss: 0.0
Loss is finite: True


In [43]:
import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

# Make sure model is in training mode
model = model.to(device)

# Freeze backbone initially
for param in model.backbone.parameters():
    param.requires_grad = False

# Classifier remains trainable
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.backbone.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.classifier.parameters(),
            "lr": 1e-3
        }
    ],
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5
)

# Modern PyTorch AMP
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

print("Device:", device)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

Device: cuda
Trainable parameters: 793100


In [44]:
def evaluate_model(model, loader):
    model.eval()

    all_probs = []
    all_targets = []

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for x, y, w, mask in loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            w = w.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type=device.type,
                enabled=(device.type == "cuda")
            ):
                logits = model(x)

            loss = weighted_masked_bce(
                logits,
                y,
                w,
                mask
            )

            total_loss += loss.item()
            total_batches += 1

            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    aucs = {}

    for j, target in enumerate(TARGETS):
        y_true = all_targets[:, j]
        y_prob = all_probs[:, j]

        # Validation is gold-only, so all should normally be present
        if len(np.unique(y_true)) >= 2:
            aucs[target] = roc_auc_score(
                y_true,
                y_prob
            )
        else:
            aucs[target] = np.nan

    valid_aucs = [
        v for v in aucs.values()
        if not np.isnan(v)
    ]

    macro_auc = np.mean(valid_aucs)

    return (
        total_loss / max(total_batches, 1),
        macro_auc,
        aucs
    )

In [45]:
NUM_EPOCHS = 5

best_auc = -np.inf
best_state = None

history = []

checkpoint_path = "/kaggle/working/best_knee_model.pt"

for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()

    # -----------------------------------------
    # Unfreeze backbone after first epoch
    # -----------------------------------------
    if epoch == 1:
        for param in model.backbone.parameters():
            param.requires_grad = True

        print("\nBackbone unfrozen for fine-tuning.")

    model.train()

    running_loss = 0.0
    num_batches = 0

    for x, y, w, mask in train_loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        w = w.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            logits = model(x)

            loss = weighted_masked_bce(
                logits,
                y,
                w,
                mask
            )

        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        num_batches += 1

    train_loss = running_loss / max(num_batches, 1)

    # -----------------------------------------
    # Validation
    # -----------------------------------------
    val_loss, val_auc, val_aucs = evaluate_model(
        model,
        val_loader
    )

    scheduler.step()

    elapsed = time.time() - epoch_start

    print("\n" + "=" * 70)
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Macro AUC: {val_auc:.4f}")
    print(f"Time       : {elapsed / 60:.2f} min")

    print("\nPer-target AUC:")
    for target, auc in val_aucs.items():
        if np.isnan(auc):
            print(f"{target:20s}: NaN")
        else:
            print(f"{target:20s}: {auc:.4f}")

    # -----------------------------------------
    # Save best model
    # -----------------------------------------
    if val_auc > best_auc:

        best_auc = val_auc

        best_state = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            {
                "model_state_dict": best_state,
                "val_auc": best_auc,
                "epoch": epoch + 1,
                "targets": TARGETS
            },
            checkpoint_path
        )

        print(f"\n*** NEW BEST MODEL: {best_auc:.4f} ***")
        print("Saved:", checkpoint_path)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc
    })

    # Memory cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print(f"Best validation Macro AUC: {best_auc:.4f}")
print(f"Checkpoint: {checkpoint_path}")


Epoch 1/5
Train Loss : 0.6208
Val Loss   : 0.6640
Val Macro AUC: 0.5376
Time       : 0.61 min

Per-target AUC:
ACL                 : 0.4074
MCL                 : 0.2222
Medial Meniscus     : 0.6714
Lateral Meniscus    : 0.3750
Medial OA           : 0.6719
Lateral OA          : NaN
PF OA               : 0.6406
Effusion            : 0.6250
Synovitis           : 0.5625
Baker's             : 0.7000
Contusion           : 0.5000
Fracture            : NaN

*** NEW BEST MODEL: 0.5376 ***
Saved: /kaggle/working/best_knee_model.pt

Backbone unfrozen for fine-tuning.

Epoch 2/5
Train Loss : 0.5624
Val Loss   : 0.6561
Val Macro AUC: 0.5115
Time       : 0.78 min

Per-target AUC:
ACL                 : 0.1852
MCL                 : 0.2222
Medial Meniscus     : 0.7143
Lateral Meniscus    : 0.3750
Medial OA           : 0.4375
Lateral OA          : NaN
PF OA               : 0.7812
Effusion            : 0.6250
Synovitis           : 0.3750
Baker's             : 0.7000
Contusion           : 0.7000
Fracture

In [47]:
checkpoint = torch.load(
    "/kaggle/working/best_knee_model.pt",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Loaded best checkpoint")
print("Best epoch:", checkpoint["epoch"])
print("Best validation AUC:", checkpoint["val_auc"])

Loaded best checkpoint
Best epoch: 3
Best validation AUC: 0.5794444444444445


In [48]:
def load_study(
    study_id,
    series_df,
    base_dir,
    series_folder,
    max_slices=SLICES_PER_SERIES
):
    df = series_df[
        series_df["StudyInstanceUID"] == study_id
    ].copy()

    output = []

    for plane in PLANE_ORDER:

        candidates = df[
            df["Anatomical_Plane"].astype(str).str.lower()
            == plane.lower()
        ]

        if len(candidates) == 0:
            output.append(
                np.zeros(
                    (SLICES_PER_SERIES, IMG_SIZE, IMG_SIZE),
                    dtype=np.float32
                )
            )
            continue

        # Prefer fluid-sensitive sequences
        fluid = candidates[
            candidates["Fluid_Sensitive"] == 1
        ]

        if len(fluid) > 0:
            candidates = fluid

        row = candidates.iloc[0]

        series_id = row["SeriesInstanceUID"]

        series_path = (
            base_dir
            / series_folder
            / study_id
            / series_id
        )

        volume = load_series(
            series_path,
            max_slices=max_slices
        )

        if volume is None:
            output.append(
                np.zeros(
                    (SLICES_PER_SERIES, IMG_SIZE, IMG_SIZE),
                    dtype=np.float32
                )
            )
            continue

        volume = resize_volume(volume)

        if len(volume) < SLICES_PER_SERIES:

            pad = np.zeros(
                (
                    SLICES_PER_SERIES - len(volume),
                    IMG_SIZE,
                    IMG_SIZE
                ),
                dtype=np.float32
            )

            volume = np.concatenate(
                [volume, pad],
                axis=0
            )

        volume = volume[:SLICES_PER_SERIES]

        output.append(volume)

    return np.stack(output).astype(np.float32)

In [49]:
TEST_CACHE_DIR = Path("/kaggle/working/mri_test_cache")
TEST_CACHE_DIR.mkdir(exist_ok=True)

existing_test = {
    p.stem for p in TEST_CACHE_DIR.glob("*.npy")
}

test_ids = test["StudyInstanceUID"].astype(str).tolist()

remaining_test = [
    sid for sid in test_ids
    if sid not in existing_test
]

print("Test studies:", len(test_ids))
print("Already cached:", len(existing_test))
print("Remaining:", len(remaining_test))

Test studies: 3
Already cached: 0
Remaining: 3


In [50]:
from tqdm.auto import tqdm
import time

start = time.time()
failed_test = []

for study_id in tqdm(
    remaining_test,
    desc="Preprocessing TEST MRI"
):

    try:

        volume = load_study(
            study_id,
            test_series,
            DATA_DIR,
            "test_series"
        )

        np.save(
            TEST_CACHE_DIR / f"{study_id}.npy",
            volume.astype(np.float16)
        )

    except Exception as e:

        failed_test.append({
            "StudyInstanceUID": study_id,
            "error": str(e)
        })

elapsed = time.time() - start

print("\nTEST PREPROCESSING COMPLETE")
print("Cached:", len(list(TEST_CACHE_DIR.glob("*.npy"))))
print("Failed:", len(failed_test))
print(f"Time: {elapsed / 60:.2f} minutes")

Preprocessing TEST MRI:   0%|          | 0/3 [00:00<?, ?it/s]


TEST PREPROCESSING COMPLETE
Cached: 3
Failed: 0
Time: 0.13 minutes


In [51]:
class KneeTestDataset(Dataset):
    def __init__(self, dataframe, cache_dir):
        self.df = dataframe.reset_index(drop=True)
        self.cache_dir = Path(cache_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row["StudyInstanceUID"]

        x = np.load(
            self.cache_dir / f"{study_id}.npy"
        ).astype(np.float32)

        x = torch.from_numpy(x)

        return x, study_id


test_dataset = KneeTestDataset(
    test,
    TEST_CACHE_DIR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test studies:", len(test_dataset))
print("Test batches:", len(test_loader))

Test studies: 3
Test batches: 1


In [52]:
model.eval()

test_predictions = []
test_study_ids = []

with torch.no_grad():

    for x, study_ids in test_loader:

        x = x.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            logits = model(x)

        probs = torch.sigmoid(logits)

        test_predictions.append(
            probs.cpu().numpy()
        )

        test_study_ids.extend(
            list(study_ids)
        )

test_predictions = np.concatenate(
    test_predictions,
    axis=0
)

print("Prediction shape:", test_predictions.shape)
print("Prediction range:",
      test_predictions.min(),
      "to",
      test_predictions.max())

print("\nFirst prediction:")
display(
    pd.DataFrame(
        test_predictions,
        columns=TARGETS,
        index=test_study_ids
    )
)

Prediction shape: (3, 12)
Prediction range: 0.001279 to 0.8477

First prediction:


,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191,0.199463,0.105408,0.340820,0.119324,0.406250,0.460449,0.407471,0.214966,0.738281,0.043854,0.329590,0.034180
1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776,0.092224,0.033844,0.215820,0.041931,0.330322,0.333008,0.313232,0.114990,0.809570,0.011070,0.219360,0.005600
1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402,0.047333,0.013687,0.146118,0.018127,0.277832,0.245483,0.249634,0.067200,0.847656,0.003765,0.152344,0.001279


In [53]:
sample_submission_path = DATA_DIR / "sample_submission.csv"

sample_submission = pd.read_csv(
    sample_submission_path
)

print("Sample submission shape:",
      sample_submission.shape)

print("\nColumns:")
print(sample_submission.columns.tolist())

display(sample_submission.head())

Sample submission shape: (3, 13)

Columns:
['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [54]:
# Create submission from the official sample format

submission = sample_submission.copy()

# Map our predictions by StudyInstanceUID
prediction_df = pd.DataFrame(
    test_predictions,
    columns=TARGETS
)

prediction_df.insert(
    0,
    "StudyInstanceUID",
    test_study_ids
)

# Replace prediction columns using StudyInstanceUID
submission = submission[
    ["StudyInstanceUID"] + TARGETS
].copy()

submission = submission.drop(
    columns=TARGETS
).merge(
    prediction_df,
    on="StudyInstanceUID",
    how="left"
)

# Restore exact sample column order
submission = submission[
    sample_submission.columns
]

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Submission saved:", submission_path)
print("Shape:", submission.shape)

display(submission)

Submission saved: /kaggle/working/submission.csv
Shape: (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.199463,0.105408,0.340820,0.119324,0.406250,0.460449,0.407471,0.214966,0.738281,0.043854,0.329590,0.034180
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.092224,0.033844,0.215820,0.041931,0.330322,0.333008,0.313232,0.114990,0.809570,0.011070,0.219360,0.005600
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.047333,0.013687,0.146118,0.018127,0.277832,0.245483,0.249634,0.067200,0.847656,0.003765,0.152344,0.001279


In [55]:
# Strict validation

assert list(submission.columns) == list(sample_submission.columns)

assert len(submission) == len(sample_submission)

assert submission["StudyInstanceUID"].equals(
    sample_submission["StudyInstanceUID"]
)

assert submission[TARGETS].notna().all().all()

assert np.isfinite(
    submission[TARGETS].values
).all()

assert (
    (submission[TARGETS].values >= 0).all()
    and
    (submission[TARGETS].values <= 1).all()
)

print("✓ Column order correct")
print("✓ Row count correct")
print("✓ StudyInstanceUID order correct")
print("✓ No missing predictions")
print("✓ All predictions finite")
print("✓ All probabilities in [0, 1]")
print()
print("READY FOR SUBMISSION")

✓ Column order correct
✓ Row count correct
✓ StudyInstanceUID order correct
✓ No missing predictions
✓ All predictions finite
✓ All probabilities in [0, 1]

READY FOR SUBMISSION
